# Baseline - модель на основе популярности
Базовое решение для задачи предсказания следующей покупки пользователя в категории одежды, обуви и аксессуаров.  
Рекомендация самых популярных товаров

In [1]:
import os
import sys
from datetime import date, timedelta
from pathlib import Path
from tqdm import tqdm

import numpy as np
import pandas as pd
import dask.dataframe as dd

sys.path.append("../src")

from data_loader import (
    download_dataset,
    load_orders,
    load_interactions,
    load_items,
)

## Загрузка данных

In [2]:
download_dataset()

Path("../models").mkdir(parents=True, exist_ok=True)
Path("../data/interim").mkdir(parents=True, exist_ok=True)

Датасет загружен ранее


In [3]:
orders_df = load_orders(train=True)
orders_df.head()

,item_id,user_id,created_timestamp,last_status,last_status_timestamp,created_date
0,15535,3423970,2025-01-11 19:20:02.550,delivered_orders,2025-01-16 10:53:03,2025-01-11
1,15535,1207601,2025-03-25 13:11:48.240,canceled_orders,2025-03-29 08:32:43,2025-03-25
2,119241,2340021,2025-05-03 20:46:18.146,canceled_orders,2025-05-12 17:10:27,2025-05-03
3,119241,2356390,2025-05-01 17:53:00.716,delivered_orders,2025-05-14 14:36:20,2025-05-01
4,119241,1490080,2025-04-22 22:59:11.160,canceled_orders,2025-04-26 06:27:28,2025-04-22


In [4]:
tracker_df = load_interactions(train=True)
tracker_df.head()

,item_id,user_id,timestamp,action_type,action_widget,date
0,678602,4211601,2025-05-05 18:11:15,page_view,cart.cartSplit,2025-05-05
1,678602,4211601,2025-05-05 18:12:57,view_description,pdp.characteristics,2025-05-05
2,678602,4211601,2025-05-05 18:12:52,view_description,pdp.richContent,2025-05-05
3,678602,4211601,2025-05-05 10:39:08,page_view,cart.cartSplit,2025-05-05
4,678602,4211601,2025-05-05 18:12:57,view_description,pdp.characteristics,2025-05-05


In [5]:
orders_ddf_test = load_orders(train=False)
orders_ddf_test.head()

,item_id,user_id,created_timestamp,last_status,last_status_timestamp,created_date
0,7510328,981,2025-07-02 08:27:17.686,delivered_orders,2025-07-06 16:14:12,2025-07-02
1,136809690,4670,2025-07-02 17:17:30.423,canceled_orders,2025-07-06 19:35:07,2025-07-02
2,232878073,12341,2025-07-02 15:17:46.393,canceled_orders,2025-07-11 15:36:40,2025-07-02
3,42322137,12860,2025-07-02 00:40:52.503,delivered_orders,2025-07-04 19:44:48,2025-07-02
4,22480781,15330,2025-07-02 07:25:11.226,delivered_orders,2025-07-03 20:07:33,2025-07-02


In [6]:
test_users = orders_ddf_test["user_id"].unique().compute().sort_values().to_list()
print(f"test_users: {len(test_users):,}")
test_users[:5]

test_users: 346,759


[1, 21, 30, 31, 51]

## Построение модели на основе популярности

In [ ]:
# ------------------------------------------------------------
# SPLITS
# ------------------------------------------------------------
TRAIN_START = date(2025, 4, 18)
TRAIN_END   = date(2025, 6, 17)

VAL_START   = date(2025, 6, 18)
VAL_END     = date(2025, 7, 1)

NEW_TRAIN_START = date(2025, 5, 2)
NEW_TRAIN_END   = date(2025, 7, 1)

# Оставляем только товары, активные в последние N дней
ACTIVE_ITEMS_DAYS = 3

# Сколько популярных товаров брать в baseline
TOPK_ITEMS = 1000

# Длина итоговой рекомендации
RECS_TOPK = 100


def filter_tracker_ddf(
    tracker_ddf: dd.DataFrame,
    start_date: date,
    end_date: date,
    keep_date: bool = False,
) -> dd.DataFrame:
    tracker_ddf = tracker_ddf.assign(
        date=dd.to_datetime(tracker_ddf["date"], format="%Y-%m-%d")
    )

    filtered = tracker_ddf[
        (tracker_ddf["date"] >= pd.to_datetime(start_date))
        & (tracker_ddf["date"] <= pd.to_datetime(end_date))
    ]

    cols = ["user_id", "item_id", "action_type"]
    if keep_date:
        cols.append("date")
    return filtered[cols]


def filter_orders_ddf(
    orders_ddf: dd.DataFrame,
    start_date: date,
    end_date: date,
    keep_created_date: bool = False,
) -> dd.DataFrame:
    orders_ddf = orders_ddf.assign(
        created_date=dd.to_datetime(orders_ddf["created_date"], format="%Y-%m-%d")
    )

    filtered = orders_ddf[
        (orders_ddf["created_date"] >= pd.to_datetime(start_date))
        & (orders_ddf["created_date"] <= pd.to_datetime(end_date))
    ]

    cols = ["user_id", "item_id"]
    if keep_created_date:
        cols.append("created_date")
    return filtered[cols]


def dcg_at_k(relevance, k):
    relevance = np.asarray(relevance)[:k]
    if relevance.size == 0:
        return 0.0
    discounts = 1.0 / np.log2(np.arange(2, relevance.size + 2))
    return float(np.sum(relevance * discounts))


def ndcg_at_k(recommended_items, relevant_items, k=100):
    if not relevant_items:
        return None

    relevance = [1 if item in relevant_items else 0 for item in recommended_items[:k]]
    dcg = dcg_at_k(relevance, k)

    ideal_relevance = sorted(relevance, reverse=True)
    idcg = dcg_at_k(ideal_relevance, k)

    return dcg / idcg if idcg > 0 else 0.0

In [ ]:
def build_popularity_model(
    orders_ddf: dd.DataFrame,
    tracker_ddf: dd.DataFrame,
    end_date: date,
    active_items_days: int | None = None,
    topk: int = 200,
):
    print("Строим модель популярности...")

    # Оставляем только товары, у которых были взаимодействия
    # за последние active_items_days дней относительно end_date
    if active_items_days is not None:
        if active_items_days <= 0:
            raise ValueError("active_items_days must be > 0")

        active_start = end_date - timedelta(days=active_items_days - 1)
        active_start_ts = pd.to_datetime(active_start)

        recent_items = dd.concat(
            [
                tracker_ddf[tracker_ddf["date"] >= active_start_ts][["item_id"]],
                orders_ddf[orders_ddf["created_date"] >= active_start_ts][["item_id"]],
            ],
            axis=0,
        ).drop_duplicates()

        tracker_ddf = tracker_ddf.merge(recent_items, on="item_id", how="inner")
        orders_ddf = orders_ddf.merge(recent_items, on="item_id", how="inner")

        n_recent_items = recent_items.shape[0].compute()
        print(f"Активных товаров за последние {active_items_days} дн.: {n_recent_items:,}")

    order_pop = (
        orders_ddf["item_id"]
        .value_counts()
        .compute()
        .rename_axis("item_id")
        .reset_index(name="order_cnt")
    )

    page_views = tracker_ddf[tracker_ddf["action_type"] == "page_view"]
    view_pop = (
        page_views["item_id"]
        .value_counts()
        .compute()
        .rename_axis("item_id")
        .reset_index(name="page_view_cnt")
    )

    pop_table = order_pop.merge(view_pop, on="item_id", how="outer").fillna(0)
    pop_table["order_cnt"] = pop_table["order_cnt"].astype(np.int32)
    pop_table["page_view_cnt"] = pop_table["page_view_cnt"].astype(np.int32)

    # Базовая формула popularity
    pop_table["score"] = 3.0 * pop_table["order_cnt"] + 1.0 * pop_table["page_view_cnt"]

    pop_table = (
        pop_table
        .sort_values(["score", "item_id"], ascending=[False, True])
        .reset_index(drop=True)
    )

    pop_table["global_pop_rank"] = np.arange(1, len(pop_table) + 1, dtype=np.int32)

    top_items = pop_table["item_id"].head(topk).astype(int).tolist()

    print(f"Отобрано top-{len(top_items)} популярных товаров")
    print(pop_table.head(10))

    return top_items, pop_table


def build_user_preferences(orders_ddf, tracker_ddf, user_ids):
    print("Строим пользовательские предпочтения...")

    orders_pdf = orders_ddf.compute() if isinstance(orders_ddf, dd.DataFrame) else orders_ddf.copy()
    tracker_pdf = tracker_ddf.compute() if isinstance(tracker_ddf, dd.DataFrame) else tracker_ddf.copy()

    # Покупки
    if "created_date" in orders_pdf.columns:
        orders_pdf["created_date"] = pd.to_datetime(orders_pdf["created_date"])
        orders_pdf = orders_pdf.sort_values(["user_id", "created_date", "item_id"])
    else:
        orders_pdf = orders_pdf.sort_values(["user_id", "item_id"])

    user_purchased_items = (
        orders_pdf.groupby("user_id")["item_id"]
        .agg(lambda x: list(map(int, x)))
        .to_dict()
    )

    # Просмотры: берем именно последние 50 после сортировки по дате
    page_views_pdf = tracker_pdf[tracker_pdf["action_type"] == "page_view"].copy()

    if "date" in page_views_pdf.columns:
        page_views_pdf["date"] = pd.to_datetime(page_views_pdf["date"])
        page_views_pdf = page_views_pdf.sort_values(["user_id", "date", "item_id"])
    else:
        page_views_pdf = page_views_pdf.sort_values(["user_id", "item_id"])

    page_views_pdf = page_views_pdf.groupby("user_id", group_keys=False).tail(50)

    user_viewed_items = (
        page_views_pdf.groupby("user_id")["item_id"]
        .agg(lambda x: list(map(int, x)))
        .to_dict()
    )

    user_preferences = {}

    for user_id in tqdm(user_ids, desc="Обработка предпочтений"):
        preferences = set(user_purchased_items.get(int(user_id), []))
        preferences.update(user_viewed_items.get(int(user_id), []))
        user_preferences[int(user_id)] = list(preferences)

    return user_preferences


def generate_recommendations(user_ids, popular_items, user_preferences, topk=100):
    print("Генерируем рекомендации...")

    recommendations = {}

    for user_id in tqdm(user_ids, desc="Создание рекомендаций"):
        seen_or_bought = set(user_preferences.get(int(user_id), []))
        available_items = [item for item in popular_items if item not in seen_or_bought]

        user_recs = available_items[:topk]

        # Если после фильтрации не хватило, добиваем только новыми item'ами,
        # но НЕ возвращаем seen/bought обратно
        if len(user_recs) < topk:
            for item in popular_items:
                if item not in seen_or_bought and item not in user_recs:
                    user_recs.append(item)
                if len(user_recs) == topk:
                    break

        recommendations[int(user_id)] = user_recs[:topk]

    return recommendations


def evaluate_recommendations(recommendations: dict, user2items: dict, k: int = 100):
    ndcg_scores = []

    for user_id, recs in recommendations.items():
        relevant_items = set(user2items.get(user_id, []))
        score = ndcg_at_k(recs, relevant_items, k=k)
        if score is not None:
            ndcg_scores.append(score)

    mean_ndcg = float(np.mean(ndcg_scores)) if ndcg_scores else np.nan
    return mean_ndcg, len(ndcg_scores)


def recommendations_to_long_df(recommendations: dict) -> pd.DataFrame:
    rows = []
    for user_id, items in recommendations.items():
        for rank, item_id in enumerate(items, start=1):
            rows.append(
                {
                    "user_id": int(user_id),
                    "item_id": int(item_id),
                    "rank": int(rank),
                }
            )
    return pd.DataFrame(rows)


def maybe_build_test_gt(orders_ddf_test):
    # Если в test-файле есть item_id и это не пустые значения, можно посчитать metric.
    if "item_id" not in orders_ddf_test.columns:
        return None

    sample = orders_ddf_test["item_id"].dropna()
    try:
        sample_n = sample.shape[0].compute()
    except Exception:
        return None

    if sample_n == 0:
        return None

    user2items_test = (
        orders_ddf_test.groupby("user_id")["item_id"]
        .apply(list, meta=("item_id", "object"))
        .compute()
        .to_dict()
    )
    return {int(k): list(map(int, v)) for k, v in user2items_test.items()}

In [9]:
# ------------------------------------------------------------
# HISTORY FOR LOCAL VALIDATION
# ------------------------------------------------------------
orders_hist_val = filter_orders_ddf(
    orders_df,
    TRAIN_START,
    TRAIN_END,
    keep_created_date=True,
)

tracker_hist_val = filter_tracker_ddf(
    tracker_df,
    TRAIN_START,
    TRAIN_END,
    keep_date=True,
)

popular_items_val, pop_table_val = build_popularity_model(
    orders_ddf=orders_hist_val,
    tracker_ddf=tracker_hist_val,
    end_date=TRAIN_END,
    active_items_days=ACTIVE_ITEMS_DAYS,
    topk=TOPK_ITEMS,
)

# ------------------------------------------------------------
# HISTORY FOR FINAL / TEST
# ------------------------------------------------------------
orders_hist_full = filter_orders_ddf(
    orders_df,
    NEW_TRAIN_START,
    NEW_TRAIN_END,
    keep_created_date=True,
)

tracker_hist_full = filter_tracker_ddf(
    tracker_df,
    NEW_TRAIN_START,
    NEW_TRAIN_END,
    keep_date=True,
)

popular_items_full, pop_table_full = build_popularity_model(
    orders_ddf=orders_hist_full,
    tracker_ddf=tracker_hist_full,
    end_date=NEW_TRAIN_END,
    active_items_days=ACTIVE_ITEMS_DAYS,
    topk=TOPK_ITEMS,
)

Строим модель популярности...
Активных товаров за последние 3 дн.: 2,275,651
Отобрано top-200 популярных товаров
     item_id  order_cnt  page_view_cnt     score  global_pop_rank
0  207631139       3242          98841  108567.0                1
1   77696741       1751          97883  103136.0                2
2   51974017       4745          88469  102704.0                3
3   48831711        794          67242   69624.0                4
4  187052809       5099          53549   68846.0                5
5  300947585        394          45989   47171.0                6
6  119058499       1091          43077   46350.0                7
7  263577203       1661          40355   45338.0                8
8  110485480        371          43788   44901.0                9
9  115409137        634          42839   44741.0               10
Строим модель популярности...
Активных товаров за последние 3 дн.: 2,233,742
Отобрано top-200 популярных товаров
     item_id  order_cnt  page_view_cnt    score 

In [10]:
# Пользователи и ground truth из future window
val_orders = filter_orders_ddf(
    orders_df,
    VAL_START,
    VAL_END,
    keep_created_date=False,
)

val_users = (
    val_orders["user_id"]
    .drop_duplicates()
    .compute()
    .sort_values()
    .tolist()
)

print(f"val_users: {len(val_users):,}")

user_preferences_val = build_user_preferences(
    orders_ddf=orders_hist_val,
    tracker_ddf=tracker_hist_val,
    user_ids=val_users,
)

val_recommendations = generate_recommendations(
    user_ids=val_users,
    popular_items=popular_items_val,
    user_preferences=user_preferences_val,
    topk=RECS_TOPK,
)

user2items_val = (
    val_orders.groupby("user_id")["item_id"]
    .apply(list, meta=("item_id", "object"))
    .compute()
    .to_dict()
)
user2items_val = {int(k): list(map(int, v)) for k, v in user2items_val.items()}

val_ndcg, val_users_cnt = evaluate_recommendations(
    recommendations=val_recommendations,
    user2items=user2items_val,
    k=RECS_TOPK,
)

print(f"Local validation NDCG@{RECS_TOPK}: {val_ndcg:.5f}")
print(f"Пользователей учтено: {val_users_cnt:,}")

val_users: 359,928
Строим пользовательские предпочтения...


Обработка предпочтений: 100%|██████████| 359928/359928 [00:04<00:00, 83842.01it/s] 


Генерируем рекомендации...


Создание рекомендаций: 100%|██████████| 359928/359928 [00:03<00:00, 90757.97it/s] 


Local validation NDCG@100: 0.00568
Пользователей учтено: 359,928


In [11]:
user_preferences_test = build_user_preferences(
    orders_ddf=orders_hist_full,
    tracker_ddf=tracker_hist_full,
    user_ids=test_users,
)

test_recommendations = generate_recommendations(
    user_ids=test_users,
    popular_items=popular_items_full,
    user_preferences=user_preferences_test,
    topk=RECS_TOPK,
)

# Если в test есть реальные item_id, считаем metric.
user2items_test = maybe_build_test_gt(orders_ddf_test)

if user2items_test is not None:
    test_ndcg, test_users_cnt = evaluate_recommendations(
        recommendations=test_recommendations,
        user2items=user2items_test,
        k=RECS_TOPK,
    )
    print(f"Test NDCG@{RECS_TOPK}: {test_ndcg:.5f}")
    print(f"Пользователей учтено: {test_users_cnt:,}")
else:
    print("Test metric skipped: в load_orders(train=False) нет доступного ground truth.")

Строим пользовательские предпочтения...


Обработка предпочтений: 100%|██████████| 346759/346759 [00:01<00:00, 182395.83it/s]


Генерируем рекомендации...


Создание рекомендаций: 100%|██████████| 346759/346759 [00:04<00:00, 76774.87it/s] 


Test NDCG@100: 0.00669
Пользователей учтено: 346,759


## Оценка качества рекомендаций

In [12]:
# Таблицы popularity
pop_table_val.to_parquet("../models/popular_items_val.parquet", index=False)
pop_table_full.to_parquet("../models/popular_items_full.parquet", index=False)

# backward compatibility: основное имя для full-history popularity
pop_table_full.to_parquet("../models/popular_items.parquet", index=False)

# Сами рекомендации как candidate source для второго уровня
pop_val_recs_df = recommendations_to_long_df(val_recommendations)
pop_test_recs_df = recommendations_to_long_df(test_recommendations)

pop_val_recs_df.to_parquet("../data/interim/pop_val_recommendations.parquet", index=False)
pop_test_recs_df.to_parquet("../data/interim/pop_test_recommendations.parquet", index=False)

print("Сохранено:")
print(" - ../models/popular_items_val.parquet")
print(" - ../models/popular_items_full.parquet")
print(" - ../models/popular_items.parquet")
print(" - ../data/interim/pop_val_recommendations.parquet")
print(" - ../data/interim/pop_test_recommendations.parquet")

print("\nПример popularity table:")
display(pop_table_full.head())

print("\nПример val recommendations:")
display(pop_val_recs_df.head())

print("\nПример test recommendations:")
display(pop_test_recs_df.head())

Сохранено:
 - ../models/popular_items_val.parquet
 - ../models/popular_items_full.parquet
 - ../models/popular_items.parquet
 - ../data/interim/pop_val_recommendations.parquet
 - ../data/interim/pop_test_recommendations.parquet

Пример popularity table:


,item_id,order_cnt,page_view_cnt,score,global_pop_rank
0,51974017,4509,83521,97048.0,1
1,207631139,2346,73721,80759.0,2
2,77696741,1456,74072,78440.0,3
3,187052809,4971,54738,69651.0,4
4,48831711,724,64637,66809.0,5



Пример val recommendations:


,user_id,item_id,rank
0,1,207631139,1
1,1,77696741,2
2,1,51974017,3
3,1,48831711,4
4,1,187052809,5



Пример test recommendations:


,user_id,item_id,rank
0,1,51974017,1
1,1,207631139,2
2,1,77696741,3
3,1,187052809,4
4,1,48831711,5
